Introduction

[Kubernetes](https://en.wikipedia.org/wiki/Kubernetes) aka K8S = open <u>container-orchestration system</u> for deployment, scaling and management automation<br>
*in 99% of cases container = Docker container

Why?<br>
compatibility matrix<br>
different app components require different dependency versions!<br>
new developers have to configure a lot of everything

Solution<br>
containerize each component

__How it solves the problem__

OS = kernel + software<br>
For example Ubuntu, openSUSE, CentOS all share the same kernel (Linux), differ in software<br>
Docker = software

VMs vs Docker<br>
Docker shares the same OS<br>
Docker is better in Utilization and Size => Speed<br>
VMs can run different OS

Where Docker applications reside<br>
Docker Hub = public store, everyone can contribute<br>
docker run <name> - runs a containered application<br>
Image = packaged docker application

Before:<br>
Developer prepares Application + deployment instruction<br>
Devops fails to deploy and they both figure out why

Now:<br>
Developer prepares Docker container<br>
It runs smoothly everywhere

Orchestration = auto scaling in the cluster
Kubernetes = container orchestration tool

Alternatives:
Docker Swarm
Kubernetes
Mesos

Nodes have to be physical (why?)

Master - nodes

Services:
API server - gateway для кластера
etcd - configuration service (zookeeper)
scheduler - distributes work
controller - monitors execution
container runtime - software (in most cases docker containers)
kubelet - gateway of the node

Main command
kubectl run <name> - deploy on server

YAML Reminder



Apart from command line, PODs can be defined in configuration files


```
# To run docker image from command line
kubectl run nginx_pod --image=nginx

# To run docker image from configuration file
kubectl create -f pod-def.yml

# To update POD using configuration
kubectl apply -f pod-def.yml

# To edit configuration of a running POD (the same as get / edit / apply)
kubectl edit pod nginx

# To export POD configuration into file
kubectl run nginx --image=nginx -o yaml > pod.yml

# To export POD configuration into file without actually running
kubectl run nginx --image=nginx --dry-run=client -o yaml > pod.yml
```

There are 4 obligatory fields:
```
apiVersion: v1
K8s enables versioning
kind: Pod
metadata:
free-format data
spec:
    containers:
        - name:
          image:
```

for POD it requires a containers list<br>
described by name nad image


It is suggested to edit those files using IDEs (PyCharm or VS)<br>
Visual Studio with a Kubernetes plugin. It does checks, auto-complete and show as a dictionary.

To delete POD<br>
`kubectl delete pod <name>`

To delete all PODs<br>
`kubectl delete --all pods`


POD Replication
Since Kunernetes is an orchestration tool, it must support scaling, resilience and load balancing.
ReplicaSet controller maintains the exact amount of POD instances on the cluster.

In Kubernetes there are two things:
RepliactionController (an old one)
ReplicaSet (a new one)
The only difference - ReplicaSet can control those PODs that were already created without any replication - it has a selector functionality.

Replicasets are created from configuration file.
kubectl create replicaset -f rs.yml

```
spec:
    template: full POD description
    replicas: 3
    selector:
```

Template is required even when using a selector for old PODs. Because RS needs to know how to start new instances.

Replication starts immediately
If there are more active PODs than defined, they are terminating. 
If there are less PODs, new ones are starting

To print all POD instances (including those from ReplicaSet)
`kubectl get pods`

To print all ReplicaSets
`kubectl get replicaset`

To get more details:
`kubectl describe replicaset <rs-name>`
`kubectl describe pod <pod-name>`

How to change replication for active ReplicaSets?

```
# By manually editing RS definition
kubectl edit replicaset rs

# By using replace
kubectl replace replicaset

# By using scale command
kubectl scale replicas 3 replicacset rs
```

__Deployments__<br>
Deployment = set of replicasets. It's a higher level in hierarchy.

It's the same as ReplicaSet, but has wider functionality. It allows updates on-the run. Thus it is recommended to use Deployments instead.

Definition is the same as the definition of ReplicaSet
```
apiVersion
kind: Deployment
metadata:
spec:
  replicas:
  template:
  selector:
```

To print all resources
`kubectl get all`

How to update currently active deployment?
`kubectl apply -f file`
`kubectl set image deployment nginx`
`kubectl edit deployment mydep`

Use-cases for update:<br>
upgrading a version of an app (using another Docker image)

POD scaling does not trigger Rollout

Two strategies:<br>
rolling update (default) = reset PODs one-by-one<br>
recreate = reset all PODs at once

To change - in definition file:
```
spec:
  strategy:
    type: Recreate | Rollout
```

How to manage rollout (deployment)?
`kubectl rollout status deployment mydep`
`kubectl rollout history deployment mydep`
`kubectl rollout undo deployment mydep`

__Services__

There 3 types of services:
- NodePort - for external access
- ClusterIP - for internal access
- LoadBalancer

Example Microservice Application

It's a standard Docker example.

It consists of 5 components:
-Voting Web App<br>
written on Python Flask
-Result Web App<br>
written on Node.JS
-Redis Backend<br>
for storing incoming votes
-Postgres Backend<br>
for storing resulting stats
-Worker process<br>that transfers vote from Redis to Postgres 

Web app face public network => there must be NodePort defined

There should be 4 services:
- Public services for Voting and Results web applications
- NodePort / LoadBalancer
- Private services for reading and writing Redis and Postgres
- ClusterIP

Total 9 definition files

It's better to wrap each POD into deployment. In that chase you can scale easily
kubectl scale deployment --replicas 3

By default deployment creates a ReplicaSet

To get several resources:
`kubectl get pods, svc`

Container ports
When defining PODs you also configure ports that gonna be used: 
```
spec:
    container:
        ports: 
            containerPort: 80
```

Environament variables
You can also configure environment variables (for example, for user / password) 

```
spec:
    container:
        env: 
name: 
            	value: 
name: 
value: 
```

Managed K8s Clusters

There are 3 options:
- GCP
- AWS
- Azure

Setting up cluster in AWS is the most involved process

Schema is the following<br>
you create a cluster<br>
configurate kubectl to manage the created cluster<br>
git clone all your POD definitions and deploy them on the cluster

In GCP and Azure you work in cloud shell<br>
In AWS you work locally

You don't SSH on the the Worker nodes<br>
You cannot access master node!

You can vIsually inspect the K8s resource

__New cluster configuration__

How to configure?
define a set of nodes (for example as VMs)
check OS and libraries prerequisites on each node
install Docker (or other container runtime) on each node
configure it to run as a service
install K8s tools on each node
kubeadm
kubelet
kubectl
initialize Master
run kubeadm init on the master node
add argument to define subnetwork for PODs 
(POD network will be added using 3rd party software)
add IP address for apiServer
in home directory add a config file
create a POD network by applying a configuration file
kubectl apply -f "..."
Join other nodes to a cluster
kubeadm join 

Commands can be run using kubectl on any node of the cluster.

__Vagrant__ = tool for VM automation
You can define all machine parameters in a file and run it.

